# **Carga y Exploración Inicial de Datos**

La fase inicial de cualquier proyecto de análisis de datos implica la carga y exploración de los conjuntos de datos disponibles. En este caso, se trabajó con múltiples archivos, uno por cada año desde 2012 hasta 2022, proporcionados por el Sistema Nacional de Vigilancia en Salud Pública (Sivigila) de Colombia. Aunque cada archivo correspondía a un año específico, todos compartían la misma estructura de variables, lo que permitió su consolidación en un único conjunto de datos para facilitar el análisis posterior. Este proceso de unión es esencial para garantizar que los datos estén completos y listos para su exploración y modelado.

## **Proceso de Unión de Datasets**
El primer paso consistió en cargar los datasets individuales y combinarlos en un único DataFrame. Dado que todas las variables eran consistentes entre los años, se utilizó la función pd.concat() de la librería Pandas en Python para apilar verticalmente los datos. Este enfoque aseguró que no se perdiera información y que todas las observaciones estuvieran disponibles en un solo lugar para su análisis.

In [ ]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import missingno as msno
import plotly.express as px
from scipy.stats import chi2_contingency, mannwhitneyu, kstest, ttest_ind
from plotly.offline import plot
from scipy.stats import zscore


In [ ]:
Data = pd.read_csv("Data.csv")
Data.head()

<ipython-input-172-2ab91b33fe0a>:1: DtypeWarning:

Columns (0,10,11,21,22,23,29,37,52,53,56,58,69) have mixed types. Specify dtype option on import or set low_memory=False.



,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,...,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion,COD_EVE.1
0,Datos_2014_875,2934558,875,2014-06-03,22,2014,7600107759,2,15,1,...,COLOMBIA,VCM VIF VSX,VALLE,CALI,NaN,VALLE,CALI,VALLE,CALI,NaN
1,Datos_2014_875,2934568,875,2014-03-24,13,2014,5200101457,18,17,1,...,COLOMBIA,VCM VIF VSX,NARIÑO,PASTO,NaN,NARIÑO,PASTO,NARIÑO,PASTO,NaN
2,Datos_2014_875,2934589,875,2014-08-12,33,2014,1100110146,0,17,1,...,COLOMBIA,VCM VIF VSX,BOGOTA,BOGOTA,NaN,BOGOTA,BOGOTA,BOGOTA,BOGOTA,NaN
3,Datos_2014_875,2934609,875,2014-05-18,21,2014,6800103534,1,17,1,...,COLOMBIA,VCM VIF VSX,SANTANDER,PIEDECUESTA,NaN,SANTANDER,PIEDECUESTA,SANTANDER,BUCARAMANGA,NaN
4,Datos_2014_875,2934620,875,2014-03-26,5,2014,6819000713,2,17,1,...,COLOMBIA,VCM VIF VSX,SANTANDER,LANDAZURI,NaN,SANTANDER,LANDAZURI,SANTANDER,CIMITARRA,NaN


## **Exploración Inicial**
Una vez consolidado el dataset, se realizó una exploración inicial para entender su estructura y dimensiones. Se utilizaron métodos como .shape para conocer el número de filas y columnas, .describe() para obtener estadísticas descriptivas de las variables numéricas, y .head() para visualizar las primeras filas del dataset. Estas herramientas permitieron identificar rápidamente el tamaño del conjunto de datos, la presencia de valores atípicos y la distribución general de las variables.

In [ ]:
print(f"Cantidad de registros: {Data.shape[0]}, Cantidad de variables: {Data.shape[1]}")

Cantidad de registros: 1014375, Cantidad de variables: 75


## **Limpieza Preliminar**
Durante esta etapa, se identificaron y eliminaron columnas irrelevantes o redundantes que no aportaban valor al análisis. Por ejemplo, se descartaron variables con valores constantes o que no estaban relacionadas con el objetivo del proyecto, como identificadores internos o metadatos administrativos, al igual que algunas variables redundantes. 

In [ ]:
DeleteColumns = [
    'Partición', 'COD_PRE', 'COD_EVE', 'COD_SUB', 'COD_ASE', 'AJUSTE', 'CER_DEF', 'Nombre_evento',
    'FEC_ARC_XL', 'FEC_AJU', 'FM_FUERZA', 'FM_UNIDAD', 'FM_GRADO',
    'confirmados', 'consecutive_origen', 'va_sispro', 'COD_EVE.1'
]
Data = Data.drop(columns=DeleteColumns)

In [ ]:
print(f"Filas: {Data.shape[0]}, Columnas: {Data.shape[1]}")

Filas: 1014375, Columnas: 58


## **Asignación de Tipos de Variables**

Durante la exploración inicial de los datos, se identificó un problema recurrente relacionado con la asignación incorrecta de tipos de variables. Muchas variables que deberían ser categóricas (por ejemplo, el tipo de caso o la pertenencia étnica) se cargaron inicialmente como variables numéricas, mientras que algunas variables numéricas (como la edad o el año del evento) se interpretaron incorrectamente como objetos. Este tipo de errores puede afectar significativamente el análisis posterior, ya que limita la capacidad de realizar operaciones matemáticas o agrupaciones lógicas. Para abordar este problema, se realizó un análisis variable por variable, revisando la naturaleza de cada una y asignando el tipo de dato más apropiado. 

*Nota*: Aunque se corrigieron los tipos de datos en esta etapa, es importante tener en cuenta que, dependiendo del análisis específico que se realice más adelante, podría ser necesario cambiar nuevamente el tipo de algunas variables. 

Tras la asignación correcta de los tipos de datos, se realizó un análisis descriptivo inicial utilizando la función ´data.describe()´. Este análisis proporcionó una visión general de las variables numéricas y de fecha, permitiendo identificar tendencias, rangos y valores atípicos.

In [ ]:
# Conversión a cadenas (string)
Data['CONSECUTIVE'] = Data['CONSECUTIVE'].astype(str)
Data['UNI_MED'] = Data['UNI_MED'].astype(str)
Data['nacionalidad'] = Data['nacionalidad'].astype(str)
Data['nombre_nacionalidad'] = Data['nombre_nacionalidad'].astype(str)
Data['SEXO'] = Data['SEXO'].astype(str)
Data['COD_PAIS_O'] = Data['COD_PAIS_O'].astype(str)
Data['COD_DPTO_O'] = Data['COD_DPTO_O'].astype(str)
Data['COD_MUN_O'] = Data['COD_MUN_O'].astype(str)
Data['AREA'] = Data['AREA'].astype(str)
Data['OCUPACION'] = Data['OCUPACION'].astype(str)
Data['TIP_SS'] = Data['TIP_SS'].astype(str)
Data['PER_ETN'] = Data['PER_ETN'].astype(str)
Data['GRU_POB'] = Data['GRU_POB'].astype(str)
Data['nom_grupo'] = Data['nom_grupo'].astype(str)
Data['GP_DISCAPA'] = Data['GP_DISCAPA'].astype(str)
Data['GP_DESPLAZ'] = Data['GP_DESPLAZ'].astype(str)
Data['GP_MIGRANT'] = Data['GP_MIGRANT'].astype(str)
Data['GP_CARCELA'] = Data['GP_CARCELA'].astype(str)
Data['GP_GESTAN'] = Data['GP_GESTAN'].astype(str)
Data['GP_INDIGEN'] = Data['GP_INDIGEN'].astype(str)
Data['GP_POBICFB'] = Data['GP_POBICFB'].astype(str)
Data['GP_MAD_COM'] = Data['GP_MAD_COM'].astype(str)
Data['GP_DESMOVI'] = Data['GP_DESMOVI'].astype(str)
Data['GP_PSIQUIA'] = Data['GP_PSIQUIA'].astype(str)
Data['GP_VIC_VIO'] = Data['GP_VIC_VIO'].astype(str)
Data['GP_OTROS'] = Data['GP_OTROS'].astype(str)
Data['fuente'] = Data['fuente'].astype(str)
Data['COD_PAIS_R'] = Data['COD_PAIS_R'].astype(str)
Data['COD_DPTO_R'] = Data['COD_DPTO_R'].astype(str)
Data['COD_MUN_R'] = Data['COD_MUN_R'].astype(str)
Data['COD_DPTO_N'] = Data['COD_DPTO_N'].astype(str)
Data['COD_MUN_N'] = Data['COD_MUN_N'].astype(str)
Data['TIP_CAS'] = Data['TIP_CAS'].astype(str)
Data['PAC_HOS'] = Data['PAC_HOS'].astype(str)
Data['CON_FIN'] = Data['CON_FIN'].astype(str)
Data['CBMTE'] = Data['CBMTE'].astype(str)
Data['Estado_final_de_caso'] = Data['Estado_final_de_caso'].astype(str)
Data['nom_est_f_caso'] = Data['nom_est_f_caso'].astype(str)
Data['Nom_upgd'] = Data['Nom_upgd'].astype(str)
Data['Pais_ocurrencia'] = Data['Pais_ocurrencia'].astype(str)
Data['Departamento_ocurrencia'] = Data['Departamento_ocurrencia'].astype(str)
Data['Municipio_ocurrencia'] = Data['Municipio_ocurrencia'].astype(str)
Data['Pais_residencia'] = Data['Pais_residencia'].astype(str)
Data['Departamento_residencia'] = Data['Departamento_residencia'].astype(str)
Data['Municipio_residencia'] = Data['Municipio_residencia'].astype(str)
Data['Departamento_Notificacion'] = Data['Departamento_Notificacion'].astype(str)
Data['Municipio_notificacion'] = Data['Municipio_notificacion'].astype(str)

# Conversión a fechas
Data['FEC_NOT'] = pd.to_datetime(Data['FEC_NOT'], format='%Y-%m-%d', errors='coerce')
Data['FEC_CON'] = pd.to_datetime(Data['FEC_CON'], format='%Y-%m-%d', errors='coerce')
Data['INI_SIN'] = pd.to_datetime(Data['INI_SIN'], format='%Y-%m-%d', errors='coerce')
Data['FEC_HOS'] = pd.to_datetime(Data['FEC_HOS'], format='%Y-%m-%d', errors='coerce')
Data['FEC_DEF'] = pd.to_datetime(Data['FEC_DEF'], format='%Y-%m-%d', errors='coerce')

# Conversión a numéricos (enteros)
Data['SEMANA'] = pd.to_numeric(Data['SEMANA'], errors='coerce').astype('Int64')
Data['ANO'] = pd.to_numeric(Data['ANO'], errors='coerce').astype('Int64')
Data['EDAD'] = pd.to_numeric(Data['EDAD'], errors='coerce').astype('Int64')
Data['sem_ges'] = pd.to_numeric(Data['sem_ges'], errors='coerce').astype('Int64')

In [ ]:
display(Data.describe())

,FEC_NOT,SEMANA,ANO,EDAD,sem_ges,FEC_CON,INI_SIN,FEC_HOS,FEC_DEF
count,1014375,1014375.0,1014375.0,1014375.0,25313.0,1014354,940478,155896,1419
mean,2018-06-10 22:44:42.030609664,27.001046,2017.914265,22.834042,19.614269,2018-06-06 13:03:02.386031872,2018-05-24 21:27:43.434125824,2019-01-24 10:06:41.436855296,2017-12-28 17:24:13.699788032
min,2012-01-01 00:00:00,1.0,2012.0,0.0,1.0,2007-01-01 00:00:00,1993-01-30 00:00:00,2011-11-07 00:00:00,2012-01-24 00:00:00
25%,2016-03-29 00:00:00,14.0,2016.0,11.0,10.0,2016-03-25 00:00:00,2016-03-03 00:00:00,2017-01-09 18:00:00,2015-10-06 00:00:00
50%,2018-10-05 00:00:00,27.0,2018.0,19.0,19.0,2018-10-01 00:00:00,2018-10-13 00:00:00,2019-06-27 00:00:00,2017-12-17 00:00:00
75%,2021-01-22 00:00:00,40.0,2021.0,32.0,28.0,2021-01-19 00:00:00,2021-02-17 00:00:00,2021-07-10 00:00:00,2020-07-05 00:00:00
max,2022-12-31 00:00:00,53.0,2022.0,131.0,45.0,2022-12-31 00:00:00,2022-12-31 00:00:00,2022-12-31 00:00:00,2022-12-30 00:00:00
std,NaN,14.836164,3.01495,16.901137,10.987746,NaN,NaN,NaN,NaN


- La columna `FEC_NOT` muestra la fecha en que se notificaron los casos de violencia de género. Con un total de 1,014,375 registros y sin valores faltantes, la fecha promedio de notificación es alrededor del **10 de junio de 2018**. Las fechas más antiguas y recientes de notificación son el **1 de enero de 2012** y el **31 de diciembre de 2022**, respectivamente, lo que indica que los datos abarcan un período de 11 años. La mediana (50%) revela que la mitad de los casos se notificaron antes del **5 de octubre de 2018**, lo que sugiere una distribución relativamente uniforme a lo largo del tiempo.

- La columna `SEMANA` representa la semana epidemiológica en que se notificó el caso. Con un promedio de **27 semanas** y un rango que va desde la semana 1 hasta la semana **53**, esta variable refleja la distribución de los casos a lo largo del año. La mediana de 27 semanas coincide con el promedio, lo que indica una distribución equilibrada entre las semanas del año.

- La columna ´ANO´ indica el año en que se notificó el caso. Con un promedio de **2017.91**, se observa que la mayoría de los casos se concentran en los años más recientes del período analizado (2012-2022). El valor mínimo es **2012** y el máximo es **2022**, lo que confirma que los datos cubren un período de 11 años.

- La columna `EDAD` refleja la edad de las víctimas de violencia de género. Con un promedio de **22.83** años y una desviación estándar de **16.90**, se observa una amplia variabilidad en las edades. El valor mínimo es **0 años**, lo que sugiere la presencia de casos que involucran a menores de edad, mientras que el valor máximo es **131** años, que probablemente es un error o un valor atípico que debe ser revisado. La mediana de **19 años** indica que la mitad de las víctimas tienen menos de 19 años, lo que resalta la vulnerabilidad de los jóvenes en estos casos.

- La columna `sem_ges` representa la semana de gestación en casos donde la víctima estaba embarazada. Con solo **25,313** registros (frente a más de un millón en otras columnas), esta variable tiene una gran cantidad de valores faltantes. El promedio es de **19.61 semanas**, con una mediana de 19 semanas, lo que sugiere que la mayoría de los casos ocurren en el segundo trimestre del embarazo. Los valores van desde **1 semana hasta 45 semanas**, lo que cubre todo el período de gestación.

- La columna `FEC_CON` indica la fecha en que la víctima buscó atención médica o consultó por el caso. Con un promedio alrededor del **6 de junio de 2018**, esta variable muestra una distribución similar a `FEC_NOT`, lo que sugiere que la mayoría de las víctimas buscan atención poco después de la notificación del caso. El valor mínimo es **1 de enero de 2007**, lo que podría indicar errores en los datos, mientras que el máximo es **31 de diciembre de 2022**, coincidiendo con el período analizado.

- La columna `INI_SIN` refleja la fecha en que comenzaron los síntomas o se identificó la violencia. Con un promedio alrededor del **24 de mayo de 2018**, esta variable muestra que, en general, los síntomas o la identificación de la violencia ocurren poco antes de la notificación del caso. El valor mínimo es **30 de enero de 1993**, lo que podría ser un error, y el máximo es **31 de diciembre de 2022**.

- La columna `FEC_HOS` indica la fecha en que la víctima fue hospitalizada. Con solo **155,896 registros**, esta variable tiene una gran cantidad de valores faltantes, lo que sugiere que no todas las víctimas requieren hospitalización. El promedio es alrededor del **24 de enero de 2019**, lo que indica que las hospitalizaciones tienden a ocurrir varios meses después de la notificación del caso. Los valores van desde **7 de noviembre de 2011** hasta **31 de diciembre de 2022**.

- La columna `FEC_DEF` refleja la fecha de defunción de las víctimas. Con solo **1,419 registros**, esta variable tiene una cantidad muy baja de valores no nulos, lo que indica que los desenlaces fatales son relativamente raros en este conjunto de datos. El promedio es alrededor del **28 de diciembre de 2017**, con valores que van desde **24 de enero de 2012** hasta **30 de diciembre de 2022**.

# **Análisis de Datos Faltantes**

El análisis de datos faltantes es una etapa crítica en la preparación de los datos, ya que la presencia de valores nulos puede afectar la calidad de los modelos predictivos y llevar a conclusiones erróneas. En este proyecto, se realizó un análisis exhaustivo para identificar y cuantificar los valores faltantes en todas las variables, con especial atención a las columnas categóricas, donde los valores nulos pueden estar representados de diversas formas (por ejemplo, cadenas vacías o términos como "NaN" o "NULL").

## **Validación de Valores Faltantes en Variables Categóricas**

Primero, se identificaron las columnas categóricas en el dataset utilizando el método select_dtypes(), que permite filtrar por tipo de dato. Luego, se verificó la presencia de valores faltantes en estas columnas utilizando la función isnull().sum(). Sin embargo, dado que los valores faltantes en variables categóricas a menudo se representan como cadenas vacías o términos específicos (por ejemplo, "NaN" o "NULL"), se realizó una conversión adicional para asegurar que todos estos casos fueran identificados correctamente como valores nulos. Esto se logró utilizando la función replace() de Pandas, que convierte cadenas vacías y otros valores comunes en pd.NA.

In [ ]:
# Identificar columnas categóricas
CategoricalColumns = Data.select_dtypes(include=['object', 'category']).columns

# Verificar valores faltantes en columnas categóricas
MissingCategorical = Data[CategoricalColumns].isnull().sum()
print(MissingCategorical[MissingCategorical > 0])

estrato      426004
FECHA_NTO     22835
dtype: int64


In [ ]:
# Convertir cadenas vacías y otros valores a NA
Data[CategoricalColumns] = Data[CategoricalColumns].replace(['', ' ', 'NaN', 'NULL', 'nan', 'null'], pd.NA)

# Verificar nuevamente los valores faltantes
MissingCategoricalUpdated = Data[CategoricalColumns].isnull().sum()
print(MissingCategoricalUpdated[MissingCategoricalUpdated > 0])

nacionalidad             426004
nombre_nacionalidad      426004
GRU_POB                  951130
nom_grupo                426004
estrato                  426004
fuente                   426004
COD_PAIS_R               654172
FECHA_NTO                 22835
CBMTE                   1013090
Pais_residencia          654172
Municipio_residencia          5
dtype: int64


## **Tabla de Porcentajes de Valores Faltantes**

Una vez identificados y normalizados los valores faltantes, se calculó el porcentaje de valores nulos para cada variable en el dataset. Estos porcentajes se organizaron en una tabla ordenada de mayor a menor, lo que permitió identificar rápidamente las columnas con mayor cantidad de datos faltantes.

In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
CBMTE,99.873321
FEC_DEF,99.860111
sem_ges,97.504572
GRU_POB,93.765126
FEC_HOS,84.631325
COD_PAIS_R,64.490154
Pais_residencia,64.490154
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697


## **Gráfico de Valores Faltantes**
Para visualizar la distribución de los valores faltantes, se creó un gráfico de barras que muestra el porcentaje de datos nulos por variable. Este gráfico facilita la identificación de patrones y ayuda a priorizar las columnas que requieren atención durante la limpieza de datos.

In [ ]:
import plotly.express as px

# Creación del gráfico de barras interactivo
fig = px.bar(
    MissingPercentage,
    x=MissingPercentage.index,
    y=MissingPercentage.values,
    labels={'x': 'Variables', 'y': 'Porcentaje de Valores Faltantes'},
    title='Porcentaje de Valores Faltantes por Variable',
    color=MissingPercentage.values,
    color_continuous_scale='Blues',
    text=MissingPercentage.values.round(2)
)

# Personalización del gráfico
fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Porcentaje de Valores Faltantes',
    title_font_size=24,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(
        family="Georgia",
        size=14,
        color="black"
    ),
    title_font=dict(
        family="Georgia",
        size=24,
        color="black"
    ),
    coloraxis_colorbar=dict(
        title='Porcentaje',
        titlefont=dict(family="Georgia", size=14),
        tickfont=dict(family="Georgia", size=12)
    )
)

fig.update_traces(textposition='outside')

# Mostrar el gráfico
fig.show()

El análisis reveló que algunas variables, como sem_ges (semana de gestación) y FEC_DEF (fecha de defunción), tienen un alto porcentaje de valores faltantes, lo que sugiere que no todos los casos involucran embarazos o desenlaces fatales. Por otro lado, variables como FEC_NOT (fecha de notificación) y EDAD (edad de la víctima) tienen pocos o ningún valor faltante, lo que indica que son columnas clave para el análisis. El gráfico de barras permitió visualizar estas diferencias de manera clara, facilitando la toma de decisiones sobre cómo manejar los datos faltantes en las siguientes etapas del proyecto. Existe la necesidad de analizar cada variable de manera individual para identificar el tratamiento óptimo de cada una de estas. 

### **Manejo de Valores Faltantes en CBMTE y FEC_DEF**

Las variables `CBMTE` (causa básica de muerte) y `FEC_DEF` (fecha de defunción) presentan una alta cantidad de valores faltantes, lo cual es esperado dado que no todas las víctimas de violencia de género fallecen. Para abordar este problema, se utilizó la variable CON_FIN (condición final), que indica si la víctima está viva, muerta o si su estado es desconocido. Este enfoque permitió identificar y manejar los valores faltantes de manera más precisa.

#### **Identificación de Valores Faltantes por Grupo CON_FIN**
Primero, se verificó la cantidad de valores faltantes en CBMTE y FEC_DEF según el valor de CON_FIN. Esto permitió entender cómo se distribuyen los datos faltantes entre las víctimas vivas, muertas y aquellas con estado desconocido.

In [ ]:
# Verificar valores faltantes en CBMTE por grupo CON_FIN
print("Valores faltantes en CBMTE por grupo CON_FIN:")
print(Data.groupby('CON_FIN')['CBMTE'].apply(lambda x: x.isnull().sum()))

# Verificar valores faltantes en FEC_DEF por grupo CON_FIN
print("Valores faltantes en FEC_DEF por grupo CON_FIN:")
print(Data.groupby('CON_FIN')['FEC_DEF'].apply(lambda x: x.isnull().sum()))

Valores faltantes en CBMTE por grupo CON_FIN:
CON_FIN
0        421
1    1012353
2        316
Name: CBMTE, dtype: int64
Valores faltantes en FEC_DEF por grupo CON_FIN:
CON_FIN
0        423
1    1012356
2        177
Name: FEC_DEF, dtype: int64


#### **Conversión de CON_FIN a Tipo Numérico**
Dado que CON_FIN es una variable categórica, se convirtió a tipo numérico para facilitar su manipulación. Esto también permitió identificar y manejar posibles errores en los datos, como valores no válidos.

In [ ]:
# Verificar valores únicos en CON_FIN
print("Valores únicos en CON_FIN:")
print(Data['CON_FIN'].unique())

Valores únicos en CON_FIN:
['1' '0' '2']


In [ ]:
# Convertir CON_FIN a tipo numérico
Data['CON_FIN'] = pd.to_numeric(Data['CON_FIN'], errors='coerce')

# Verificar valores únicos en CON_FIN después de la conversión
print("Valores únicos en CON_FIN después de la conversión:")
print(Data['CON_FIN'].unique())

Valores únicos en CON_FIN después de la conversión:
[1 0 2]


In [ ]:
# Verificar valores faltantes en CON_FIN
print("Valores faltantes en CON_FIN después de la conversión:")
print(Data['CON_FIN'].isnull().sum())

Valores faltantes en CON_FIN después de la conversión:
0


#### **Imputación de Valores en CBMTE y FEC_DEF**
Para las víctimas vivas (`CON_FIN` == 1), se imputó el código "0000" en la variable CBMTE, ya que este valor no corresponde a ningún código de muerte y permite identificar claramente que la víctima no falleció. De manera similar, en la variable FEC_DEF, se imputó una fecha lejana (2090-12-31) para indicar que la víctima está viva.

In [ ]:
# Imputar "0000" en CBMTE para pacientes vivos (CON_FIN == 1)
Data.loc[Data['CON_FIN'] == 1, 'CBMTE'] = Data.loc[Data['CON_FIN'] == 1, 'CBMTE'].fillna("0000")

In [ ]:
# Convertir FEC_DEF a datetime si no lo está ya
Data['FEC_DEF'] = pd.to_datetime(Data['FEC_DEF'], errors='coerce')

# Imputar "9999-12-31" en FEC_DEF para pacientes vivos (CON_FIN == 1)
fecha_no_fallecio = pd.to_datetime('2090-12-31')
Data.loc[Data['CON_FIN'] == 1, 'FEC_DEF'] = Data.loc[Data['CON_FIN'] == 1, 'FEC_DEF'].fillna(fecha_no_fallecio)

#### **Verificación de la Imputación**
Finalmente, se verificó que los valores faltantes en `CBMTE` y `FEC_DEF` se hubieran imputado correctamente para las víctimas vivas. Esto aseguró que el proceso de imputación fuera efectivo y que los datos estuvieran listos para el análisis posterior.

In [ ]:
# Verificar valores faltantes en CBMTE por grupo CON_FIN
print("Valores faltantes en CBMTE por grupo CON_FIN después de la imputación:")
print(Data.groupby('CON_FIN')['CBMTE'].apply(lambda x: x.isnull().sum()))

# Verificar valores faltantes en FEC_DEF por grupo CON_FIN
print("Valores faltantes en FEC_DEF por grupo CON_FIN después de la imputación:")
print(Data.groupby('CON_FIN')['FEC_DEF'].apply(lambda x: x.isnull().sum()))

Valores faltantes en CBMTE por grupo CON_FIN después de la imputación:
CON_FIN
0    421
1      0
2    316
Name: CBMTE, dtype: int64
Valores faltantes en FEC_DEF por grupo CON_FIN después de la imputación:
CON_FIN
0    423
1      0
2    177
Name: FEC_DEF, dtype: int64


In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
sem_ges,97.504572
GRU_POB,93.765126
FEC_HOS,84.631325
COD_PAIS_R,64.490154
Pais_residencia,64.490154
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697
estrato,41.996697
fuente,41.996697


In [ ]:
import plotly.express as px

# Creación del gráfico de barras interactivo
fig = px.bar(
    MissingPercentage,
    x=MissingPercentage.index,
    y=MissingPercentage.values,
    labels={'x': 'Variables', 'y': 'Porcentaje de Valores Faltantes'},
    title='Porcentaje de Valores Faltantes por Variable',
    color=MissingPercentage.values,
    color_continuous_scale='Blues',
    text=MissingPercentage.values.round(2)
)

# Personalización del gráfico
fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Porcentaje de Valores Faltantes',
    title_font_size=24,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(
        family="Georgia",
        size=14,
        color="black"
    ),
    title_font=dict(
        family="Georgia",
        size=24,
        color="black"
    ),
    coloraxis_colorbar=dict(
        title='Porcentaje',
        titlefont=dict(family="Georgia", size=14),
        tickfont=dict(family="Georgia", size=12)
    )
)

fig.update_traces(textposition='outside')

# Mostrar el gráfico
fig.show()

#### **Conclusión**
El manejo de valores faltantes en las variables CBMTE (causa básica de muerte) y FEC_DEF (fecha de defunción) fue fundamental para mejorar la calidad y utilidad del dataset. Tras la imputación de valores para las víctimas vivas (CON_FIN == 1), se observó una reducción significativa en el porcentaje de datos faltantes. En el caso de CBMTE, el porcentaje de valores faltantes disminuyó a 0.072656%, mientras que en FEC_DEF, el porcentaje se redujo a 0.059150%. Estos resultados confirman que la estrategia de imputación fue efectiva y que los datos están ahora en mejores condiciones para su análisis y modelado.

### **Análisis y Eliminación de la Variable GRU_POB**

La variable `GRU_POB`, que indica el grupo poblacional al que pertenece la víctima, presentó problemas significativos que justificaron su eliminación del dataset. A continuación, se detallan las razones y el proceso seguido para tomar esta decisión.

- Alto Porcentaje de Datos Faltantes: `GRU_POB` tiene un **93.77% de valores faltantes**, lo que la hace prácticamente inútil para el análisis. Según la literatura, cuando una variable tiene más del 50-60% de datos faltantes, su utilidad se reduce drásticamente y es preferible eliminarla para evitar sesgos o problemas en el modelo (Allison, 2001; Schafer & Graham, 2002).

- Inconsistencias en los Valores: Los valores únicos en GRU_POB incluyen inconsistencias, como la presencia de cadenas vacías (' '), valores duplicados en diferentes formatos (por ejemplo, '5.0' y '5'), y valores no estandarizados. Esto sugiere problemas de calidad en los datos que dificultarían su uso en el análisis.

- Redundancia con Otras Variables: `GRU_POB` está relacionada con otras columnas como `GP_GESTAN` (gestante), `GP_MIGRANT` (migrante), `GP_DISCAPA`(discapacitado), entre otras, que ya capturan información detallada sobre los grupos poblacionales. Estas columnas tienen una cobertura de datos mucho mejor y no presentan un porcentaje tan alto de valores faltantes.

#### **Proceso de Eliminación de GRU_POB**
Se procedió a eliminar la columna GRU_POB del dataset utilizando el método drop() de Pandas. Luego, se verificó que la columna hubiera sido eliminada correctamente y se revisaron los valores faltantes en las columnas relacionadas para asegurar que la información sobre los grupos poblacionales estuviera bien representada.


In [ ]:
# Eliminar la columna GRU_POB
Data = Data.drop(columns=['GRU_POB'])

# Verificar que la columna ha sido eliminada
print("Columnas después de eliminar GRU_POB:")
print(Data.columns)

Columnas después de eliminar GRU_POB:
Index(['CONSECUTIVE', 'FEC_NOT', 'SEMANA', 'ANO', 'EDAD', 'UNI_MED',
       'nacionalidad', 'nombre_nacionalidad', 'SEXO', 'COD_PAIS_O',
       'COD_DPTO_O', 'COD_MUN_O', 'AREA', 'OCUPACION', 'TIP_SS', 'PER_ETN',
       'nom_grupo', 'estrato', 'GP_DISCAPA', 'GP_DESPLAZ', 'GP_MIGRANT',
       'GP_CARCELA', 'GP_GESTAN', 'sem_ges', 'GP_INDIGEN', 'GP_POBICFB',
       'GP_MAD_COM', 'GP_DESMOVI', 'GP_PSIQUIA', 'GP_VIC_VIO', 'GP_OTROS',
       'fuente', 'COD_PAIS_R', 'COD_DPTO_R', 'COD_MUN_R', 'COD_DPTO_N',
       'COD_MUN_N', 'FEC_CON', 'INI_SIN', 'TIP_CAS', 'PAC_HOS', 'FEC_HOS',
       'CON_FIN', 'FEC_DEF', 'FECHA_NTO', 'CBMTE', 'Estado_final_de_caso',
       'nom_est_f_caso', 'Nom_upgd', 'Pais_ocurrencia',
       'Departamento_ocurrencia', 'Municipio_ocurrencia', 'Pais_residencia',
       'Departamento_residencia', 'Municipio_residencia',
       'Departamento_Notificacion', 'Municipio_notificacion'],
      dtype='object')


In [ ]:
print(f"Filas: {Data.shape[0]}, Columnas: {Data.shape[1]}")

Filas: 1014375, Columnas: 57


In [ ]:
# Verificar valores faltantes en columnas relacionadas
columnas_relacionadas = ['GP_DISCAPA', 'GP_DESPLAZ', 'GP_MIGRANT', 'GP_CARCELA',
                        'GP_GESTAN', 'GP_INDIGEN', 'GP_POBICFB', 'GP_MAD_COM',
                         'GP_DESMOVI', 'GP_PSIQUIA', 'GP_VIC_VIO', 'GP_OTROS']
print("Valores faltantes en columnas relacionadas:")
print(Data[columnas_relacionadas].isnull().sum())

Valores faltantes en columnas relacionadas:
GP_DISCAPA    0
GP_DESPLAZ    0
GP_MIGRANT    0
GP_CARCELA    0
GP_GESTAN     0
GP_INDIGEN    0
GP_POBICFB    0
GP_MAD_COM    0
GP_DESMOVI    0
GP_PSIQUIA    0
GP_VIC_VIO    0
GP_OTROS      0
dtype: int64


In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
sem_ges,97.504572
FEC_HOS,84.631325
COD_PAIS_R,64.490154
Pais_residencia,64.490154
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697
estrato,41.996697
fuente,41.996697
INI_SIN,7.284978


#### **Conclusiones**
La variable `GRU_POB` fue eliminada debido a su alto porcentaje de datos faltantes **(93.77%)**, inconsistencias en los valores y redundancia con otras columnas que capturan información similar de manera más precisa. Esta decisión se alinea con las mejores prácticas en el manejo de datos faltantes y contribuye a mejorar la calidad y utilidad del dataset para el análisis de violencia de género.

### **Manejo de Valores Faltantes en sem_ges (Semana de Gestación)**
La variable `sem_ges`, que indica la semana de gestación de las víctimas embarazadas, presentó un alto porcentaje de valores faltantes. Esto se debe a que no todas las víctimas son gestantes, y por lo tanto, esta variable solo es relevante para un subconjunto de los datos. Para abordar este problema, se utilizó la variable `GP_GESTAN`, que indica si la víctima es gestante (1) o no (2), para imputar valores adecuados en sem_ges.

#### **Proceso de Imputación en sem_ges**
1. Verificación del Tipo de Dato: Primero, se verificó el tipo de dato de sem_ges para asegurar que fuera numérico y estuviera listo para la imputación.

In [ ]:
# Verificar el tipo de dato de sem_ges
print("Tipo de dato de sem_ges:")
print(Data['sem_ges'].dtype)

Tipo de dato de sem_ges:
Int64


2. Imputación de Valores para No Gestantes: Para las víctimas no gestantes (GP_GESTAN == 2), se imputó el valor 0 en sem_ges, ya que este valor indica que no hay gestación. Esto se hizo utilizando el método fillna() de Pandas.

In [ ]:
# Imputar 0 en sem_ges para no gestantes (GP_GESTAN == 2)
Data.loc[Data['GP_GESTAN'] == 2, 'sem_ges'] = Data.loc[Data['GP_GESTAN'] == 2, 'sem_ges'].fillna(0)

3. Verificación de Valores Faltantes Después de la Imputación: Después de la imputación, se verificó que no hubiera valores faltantes en sem_ges para las víctimas no gestantes.

In [ ]:
# Verificar valores faltantes en sem_ges después de la imputación
print("Valores faltantes en sem_ges después de la imputación:")
print(Data['sem_ges'].isnull().sum())

# Verificar valores únicos en sem_ges
print("Valores únicos en sem_ges después de la imputación:")
print(Data['sem_ges'].unique())

Valores faltantes en sem_ges después de la imputación:
989062
Valores únicos en sem_ges después de la imputación:
<IntegerArray>
[<NA>,    3,   26,    7,   19,   32,   36,   30,    9,   34,   22,   14,   23,
    6,   31,   21,   27,   24,   18,    2,   25,   10,   12,   11,    5,   16,
   15,   29,   38,   35,   28,   17,    8,   20,   37,   39,   40,    1,   13,
    4,   33,   44,   45,   41,   42,   43]
Length: 46, dtype: Int64


5. Verificación de Gestantes con sem_ges Faltante: Se identificaron las víctimas gestantes (GP_GESTAN == 1) que aún tenían valores faltantes en sem_ges después de la imputación. Esto permitió confirmar que los valores faltantes restantes correspondían únicamente a gestantes.

In [ ]:
# Verificar gestantes con sem_ges faltante después de la imputación
gestantes_con_sem_ges_faltante = Data[(Data['GP_GESTAN'] == 1) & (Data['sem_ges'].isna())]
print("Gestantes con sem_ges faltante después de la imputación:")
print(gestantes_con_sem_ges_faltante[['GP_GESTAN', 'sem_ges']])

6. Verificación de No Gestantes con sem_ges Faltante: Se confirmó que no había víctimas no gestantes (GP_GESTAN == 2) con valores faltantes en sem_ges después de la imputación.

In [ ]:
# Verificar no gestantes con sem_ges faltante después de la imputación
no_gestantes_con_sem_ges_faltante = Data[(Data['GP_GESTAN'] == '2') & (Data['sem_ges'].isna())]
print("No gestantes con sem_ges faltante después de la imputación:")
print(no_gestantes_con_sem_ges_faltante[['GP_GESTAN', 'sem_ges']])

No gestantes con sem_ges faltante después de la imputación:
Empty DataFrame
Columns: [GP_GESTAN, sem_ges]
Index: []


In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
FEC_HOS,84.631325
COD_PAIS_R,64.490154
Pais_residencia,64.490154
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697
estrato,41.996697
fuente,41.996697
INI_SIN,7.284978
sem_ges,2.667652


In [ ]:
import plotly.express as px

# Creación del gráfico de barras interactivo
fig = px.bar(
    MissingPercentage,
    x=MissingPercentage.index,
    y=MissingPercentage.values,
    labels={'x': 'Variables', 'y': 'Porcentaje de Valores Faltantes'},
    title='Porcentaje de Valores Faltantes por Variable',
    color=MissingPercentage.values,
    color_continuous_scale='Blues',
    text=MissingPercentage.values.round(2)
)

# Personalización del gráfico
fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Porcentaje de Valores Faltantes',
    title_font_size=24,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(
        family="Georgia",
        size=14,
        color="black"
    ),
    title_font=dict(
        family="Georgia",
        size=24,
        color="black"
    ),
    coloraxis_colorbar=dict(
        title='Porcentaje',
        titlefont=dict(family="Georgia", size=14),
        tickfont=dict(family="Georgia", size=12)
    )
)

fig.update_traces(textposition='outside')

# Mostrar el gráfico
fig.show()

### **Manejo de Valores Faltantes en FEC_HOS (Fecha de Hospitalización)**
La variable `FEC_HOS`, que indica la fecha en que la víctima fue hospitalizada, presentó una cantidad significativa de valores faltantes. Esto se debe a que no todas las víctimas de violencia de género requieren hospitalización. Para abordar este problema, se utilizó la variable `PAC_HOS`, que indica si la víctima fue hospitalizada (1) o no (2), para imputar valores adecuados en `FEC_HOS`.

#### **Proceso de Imputación en FEC_HOS**

1. Verificación del Tipo de Dato: Primero, se verificó el tipo de dato de `FEC_HOS` para asegurar que fuera de tipo datetime y estuviera listo para la imputación. Se revisó la cantidad de valores faltantes en `FEC_HOS` antes de la imputación.

In [ ]:
# Verificar el tipo de dato de FEC_HOS
print("Tipo de dato de FEC_HOS:")
print(Data['FEC_HOS'].dtype)

# Verificar valores faltantes en FEC_HOS
print("Valores faltantes en FEC_HOS:")
print(Data['FEC_HOS'].isnull().sum())

Tipo de dato de FEC_HOS:
datetime64[ns]
Valores faltantes en FEC_HOS:
858479


2. Imputación de Valores para No Hospitalizados: Para las víctimas no hospitalizadas (`PAC_HOS` == 2), se imputó una fecha lejana (2099-12-31) en `FEC_HOS`, ya que este valor indica que no hubo hospitalización. Esto se hizo utilizando el método fillna() de Pandas.  Después de la imputación, se verificó que no hubiera valores faltantes en `FEC_HOS` para las víctimas no hospitalizadas.

In [ ]:
# Imputar "2099-12-31" en FEC_HOS para no hospitalizados (PAC_HOS == '2')
fecha_no_hospitalizado = pd.to_datetime('2099-12-31')
Data.loc[Data['PAC_HOS'] == '2', 'FEC_HOS'] = Data.loc[Data['PAC_HOS'] == '2', 'FEC_HOS'].fillna(fecha_no_hospitalizado)

# Verificar valores faltantes en FEC_HOS después de la imputación
print("Valores faltantes en FEC_HOS después de la imputación:")
print(Data['FEC_HOS'].isnull().sum())

Valores faltantes en FEC_HOS después de la imputación:
2


In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
COD_PAIS_R,64.490154
Pais_residencia,64.490154
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697
estrato,41.996697
fuente,41.996697
INI_SIN,7.284978
sem_ges,2.667652
FECHA_NTO,2.251140


In [ ]:
# Verificar valores únicos en PAC_HOS
print("Valores únicos en PAC_HOS:")
print(Data['PAC_HOS'].unique())

In [ ]:
import plotly.express as px

# Creación del gráfico de barras interactivo
fig = px.bar(
    MissingPercentage,
    x=MissingPercentage.index,
    y=MissingPercentage.values,
    labels={'x': 'Variables', 'y': 'Porcentaje de Valores Faltantes'},
    title='Porcentaje de Valores Faltantes por Variable',
    color=MissingPercentage.values,
    color_continuous_scale='Blues',
    text=MissingPercentage.values.round(2)
)

# Personalización del gráfico
fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Porcentaje de Valores Faltantes',
    title_font_size=24,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(
        family="Georgia",
        size=14,
        color="black"
    ),
    title_font=dict(
        family="Georgia",
        size=24,
        color="black"
    ),
    coloraxis_colorbar=dict(
        title='Porcentaje',
        titlefont=dict(family="Georgia", size=14),
        tickfont=dict(family="Georgia", size=12)
    )
)

fig.update_traces(textposition='outside')

# Mostrar el gráfico
fig.show()

### **Manejo de Valores Faltantes en COD_PAIS_R y Pais_residencia**

Las variables `COD_PAIS_R` (código del país de residencia) y `Pais_residencia` (nombre del país de residencia) presentaron una cantidad significativa de valores faltantes. Sin embargo, en muchos casos, se disponía de información en la columna `Departamento_residencia`, lo que permitió imputar valores para las víctimas que residen en Colombia. A continuación, se describe el proceso seguido para manejar estos valores faltantes.

#### **Proceso de Imputación en COD_PAIS_R y Pais_residencia**

1. Verificación de Valores Únicos en Departamento_residencia: Primero, se verificaron los valores únicos en `Departamento_residencia` para identificar los departamentos válidos de Colombia y otros casos especiales, como "EXTERIOR" o "PROCEDENCIA DESCONOCIDA".

In [ ]:
# Verificar valores únicos en Departamento_residencia
print("Valores únicos en Departamento_residencia:")
print(Data['Departamento_residencia'].unique())

Valores únicos en Departamento_residencia:
['VALLE' 'NARIÑO' 'BOGOTA' 'SANTANDER' 'ANTIOQUIA' 'BOLIVAR' 'HUILA'
 'CAQUETA' 'BOYACA' 'CAUCA' 'META' 'GUAJIRA' 'CESAR' 'CUNDINAMARCA'
 'TOLIMA' 'ATLANTICO' 'CALDAS' 'QUINDIO' 'GUAVIARE' 'SUCRE' 'MAGDALENA'
 'PUTUMAYO' 'RISARALDA' 'CORDOBA' 'VICHADA' 'NORTE SANTANDER' 'CASANARE'
 'ARAUCA' 'AMAZONAS' 'GUAINIA' 'CHOCO' 'EXTERIOR' 'SAN ANDRES'
 'PROCEDENCIA DESCONOCIDA' 'VAUPES']


2. Lista de Departamentos Válidos de Colombia: Se creó una lista con los nombres de los departamentos de Colombia para identificar las víctimas que residen en el país.

In [ ]:
# Lista de departamentos válidos de Colombia
departamentos_colombia = [
    'VALLE', 'MARINO', 'BOGOTA', 'SANTANDER', 'ANTIOQUIA', 'BOLIVAR', 'HUILA',
    'CAQUETA', 'BOYACA', 'CAUCA', 'META', 'GUAJIRA', 'CESAR', 'CUNDINAMARCA',
    'TOLIMA', 'ATLANTICO', 'CALDAS', 'QUINDIO', 'GUAVIARE', 'SUCRE', 'MAGDALEMA',
    'PUTUNAVO', 'RISARAIDA', 'CORDOBA', 'VICHADA', 'MORTE SANTANDER', 'CASAWARE',
    'ARAUCA', 'AMAZONAS', 'GUATNIA', 'CHOCO', 'SAN ANDRES', 'VAUPES'
]

3. Imputación de Valores para Residentes en Colombia: Para las víctimas cuyo Departamento_residencia es un departamento válido de Colombia y que tenían valores faltantes en `COD_PAIS_R` o `Pais_residencia`, se imputó el código 170 y el nombre COLOMBIA, respectivamente.

In [ ]:
# Código y nombre de Colombia (como cadenas)
codigo_colombia = "170"  # Asegúrate de que sea una cadena
pais_colombia = "COLOMBIA"

# Condición: Departamento_residencia es un departamento de Colombia y COD_PAIS_R/Pais_residencia son nulos
condicion = (Data['Departamento_residencia'].isin(departamentos_colombia)) & (Data['COD_PAIS_R'].isna() | Data['Pais_residencia'].isna())

# Imputar
Data.loc[condicion, 'COD_PAIS_R'] = codigo_colombia
Data.loc[condicion, 'Pais_residencia'] = pais_colombia

4. Manejo de Casos Especiales: Para los casos en que Departamento_residencia es "EXTERIOR" o "PROCEDENCIA DESCONOCIDA", se dejaron los valores faltantes en  `COD_PAIS_R` y `Pais_residencia`, ya que no se puede inferir el país de residencia.

In [ ]:
# Condición: Departamento_residencia es EXTERIOR o PROCEDENCIA DESCONOCIDA
condicion_exterior_desconocido = Data['Departamento_residencia'].isin(['EXTERIOR', 'PROCEDENCIA DESCONOCIDA'])

# Dejar los valores faltantes en COD_PAIS_R y Pais_residencia
Data.loc[condicion_exterior_desconocido, 'COD_PAIS_R'] = pd.NA
Data.loc[condicion_exterior_desconocido, 'Pais_residencia'] = pd.NA

5. Manejo de Departamentos No Válidos: Para los casos en que `Departamento_residencia` no es un departamento válido de Colombia ni un caso especial, se dejaron los valores faltantes en `COD_PAIS_R` y `Pais_residencia`.

In [ ]:
# Condición: Departamento_residencia no es un departamento de Colombia y no es EXTERIOR o PROCEDENCIA DESCONOCIDA
condicion_faltante_o_no_valido = (
    ~Data['Departamento_residencia'].isin(departamentos_colombia) &
    Data['Departamento_residencia'].notna() &
    ~Data['Departamento_residencia'].isin(['EXTERIOR', 'PROCEDENCIA DESCONOCIDA'])
)

# Dejar los valores faltantes en COD_PAIS_R y Pais_residencia
Data.loc[condicion_faltante_o_no_valido, 'COD_PAIS_R'] = pd.NA
Data.loc[condicion_faltante_o_no_valido, 'Pais_residencia'] = pd.NA

6. Verificación de Valores Faltantes Después de la Imputación: Finalmente, se verificó la cantidad de valores faltantes en COD_PAIS_R y Pais_residencia después de la imputación.

In [ ]:
print("Valores faltantes en COD_PAIS_R después de la imputación:")
print(Data['COD_PAIS_R'].isnull().sum())

print("Valores faltantes en Pais_residencia después de la imputación:")
print(Data['Pais_residencia'].isnull().sum())

Valores faltantes en COD_PAIS_R después de la imputación:
86917
Valores faltantes en Pais_residencia después de la imputación:
86917


In [ ]:
import pandas as pd

# Calcular el porcentaje de valores faltantes por columna
MissingPercentage = (Data.isnull().sum() / len(Data)) * 100
MissingPercentage = MissingPercentage[MissingPercentage > 0]  # Solo columnas con datos faltantes
MissingPercentage = MissingPercentage.sort_values(ascending=False)  # Ordenar de mayor a menor

display((MissingPercentage))

,0
nacionalidad,41.996697
nombre_nacionalidad,41.996697
nom_grupo,41.996697
estrato,41.996697
fuente,41.996697
COD_PAIS_R,8.568527
Pais_residencia,8.568527
INI_SIN,7.284978
sem_ges,2.667652
FECHA_NTO,2.251140


In [ ]:
import plotly.express as px

# Creación del gráfico de barras interactivo
fig = px.bar(
    MissingPercentage,
    x=MissingPercentage.index,
    y=MissingPercentage.values,
    labels={'x': 'Variables', 'y': 'Porcentaje de Valores Faltantes'},
    title='Porcentaje de Valores Faltantes por Variable',
    color=MissingPercentage.values,
    color_continuous_scale='Blues',
    text=MissingPercentage.values.round(2)
)

# Personalización del gráfico
fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Porcentaje de Valores Faltantes',
    title_font_size=24,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(
        family="Georgia",
        size=14,
        color="black"
    ),
    title_font=dict(
        family="Georgia",
        size=24,
        color="black"
    ),
    coloraxis_colorbar=dict(
        title='Porcentaje',
        titlefont=dict(family="Georgia", size=14),
        tickfont=dict(family="Georgia", size=12)
    )
)

fig.update_traces(textposition='outside')

# Mostrar el gráfico
fig.show()